## cek file nonformal

In [1]:
import pandas as pd

df = pd.read_csv("NEWS_NONFORMAL1.csv")

df["nomor"] = range(len(df))

cols = ["nomor"] + [col for col in df.columns if col != "nomor"]
df = df[cols]

df

,nomor,text
0,0,Gubernur Banten Terpilih Andra Soni Temui Pres...
1,1,Remaja Ditindak Polisi di Serpong karena Bawa ...
2,2,LPSK Siap Berikan Perlindungan Saksi dan Korba...
3,3,Polisi Akan Olah TKP Kebakaran Glodok Plaza Se...
4,4,Polda Riau Tangkap 2 Kurir Jaringan Internasio...
...,...,...
42818,42818,"Dalam menjalankan di bulan Ramadhan, ada beber..."
42819,42819,"Panglima TNI Laksamana TNI Yudo Margono, S.E.,..."
42820,42820,MNC Portal Indonesia berkesempatan menggelar a...
42821,42821,Pengamat politik Ikhwan Arif mengatakan dukung...


In [4]:
df.info()
df.to_csv("NEWS_NONFORMAL1_new.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42823 entries, 0 to 42822
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   nomor   42823 non-null  int64 
 1   text    42823 non-null  object
dtypes: int64(1), object(1)
memory usage: 669.2+ KB


## **GENERATE**

In [ ]:
# =========================
# CELL 1 — SETUP ENV
# =========================

from openai import OpenAI
import pandas as pd
import os
import time

# API
API_KEY = "xxxx"

# MODEL
MODEL_NAME = "deepseek-ai/DeepSeek-V4-Pro"

# CLIENT
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=API_KEY,
)

# PROMPT
SYSTEM_PROMPT = """
"Role:
Kamu adalah netizen Indonesia yang aktif di media sosial (X/Twitter, TikTok, atau Instagram). Kamu suka sharing berita terbaru dengan gaya ""nge-spill"" atau bercerita ke teman tongkrongan. Gaya bicaramu santai, informatif, dan sedikit yapping (bercerita panjang lebar), tapi tetap sopan.

Tugas:
Ceritakan kembali teks formal yang diberikan menjadi bahasa media sosial yang sangat natural. Gunakan alur bercerita, bukan sekadar mengganti kata per kata.

Aturan Ketat:
1. JANGAN menambah/menghapus fakta: Nama, jabatan, instansi, angka, tanggal, lokasi, kronologi, dan pihak terlibat harus tetap ada.
2. JANGAN mengubah status hukum: Kata ""diduga"", ""disebut"", ""menurut"", atau ""diklaim"" harus tetap dipertahankan konteksnya. Jangan menghakimi tersangka seolah sudah pasti bersalah.
3. TARGET PANJANG: Hasil rewrite harus 95%–115% dari panjang teks asli. Jangan diringkas/dipadatkan. Tetap masukkan detail kutipan tapi ubah menjadi gaya bahasa lisan.
4. GAYA BAHASA:
   - Pakai bahasa Indonesia non-formal/percakapan: gak, udah, aja, banget, emang, kayak, gitu, sih, nih, kok, deh, dong, lah, ya.
   - Gunakan kata sambung natural: ""Nah jadi tuh"", ""Btw"", ""Terus ya"", ""Jujur sih"", ""Gini lho"", ""Katanya"", ""Gak cuma itu"", dan lain-lain yang biasa dipakai netizen indonesia.
   - Boleh campur menggunakan singkatan ala netizen indonesia seperti: yg, utk, dgn, krn, ttg, jg, atau yang lain.
   - Hindari struktur kalimat pasif koran yang kaku (contoh: ""Diterangkan oleh Budi"" ganti jadi ""Si Budi sempet terangin kalau..."").
5.  Dilarang menggunakan umpatan, kata kasar, atau membuat teks jadi jauh lebih provokatif/hate speech.

Output:
Langsung tulis hasil rewrite saja tanpa pembukaan atau penjelasan apapun."		
"""

In [9]:
# =========================
# CELL 2 — SETUP INPUT OUTPUT
# =========================

# INPUT OUTPUT
INPUT_CSV = "NEWS_NONFORMAL1_new.csv"
OUTPUT_CSV = "news_nonformal.csv"

# DELAY
DELAY_SECONDS = 1

# LOAD INPUT
df = pd.read_csv(INPUT_CSV)

# CREATE OUTPUT FILE IF NOT EXISTS
if not os.path.exists(OUTPUT_CSV):

    output_df = pd.DataFrame(columns=[
        "index",
        "text_ori",
        "text_sintesa"
    ])

    output_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

# LOAD EXISTING OUTPUT
existing_df = pd.read_csv(OUTPUT_CSV)

# INDEX YANG SUDAH DIPROSES
processed_indexes = set(existing_df["index"].tolist())

print(f"Loaded input data: {len(df)}")
print(f"Already processed: {len(processed_indexes)}")

Loaded input data: 42823
Already processed: 2


In [ ]:
# =========================
# CELL 3 — RUN GENERATION
# =========================

# SET RANGE NOMOR
START_INDEX = 42000
END_INDEX = 42100

# FILTER BERDASARKAN KOLOM NOMOR
subset_df = df[
    (df["nomor"] >= START_INDEX) &
    (df["nomor"] < END_INDEX)
]

print(f"Processing nomor {START_INDEX} sampai {END_INDEX}")

for _, row in subset_df.iterrows():

    nomor = int(row["nomor"])

    # SKIP JIKA SUDAH ADA
    if nomor in processed_indexes:
        print(f"[SKIP] {nomor} already processed")
        continue

    text = str(row["text"]).strip()

    # SKIP EMPTY
    if text == "" or text.lower() == "nan":
        print(f"[SKIP] {nomor} empty")
        continue

    try:

        print(f"\n[PROCESSING] Nomor {nomor}")

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": text
                }
            ],
            temperature=0.8,
            max_tokens=4096
        )

        generated_text = (
            response.choices[0]
            .message.content
            .strip()
            .replace("\n", " ")
        )

        # SAVE LANGSUNG
        result_df = pd.DataFrame([{
            "index": nomor,
            "text_ori": text,
            "text_sintesa": generated_text
        }])

        result_df.to_csv(
            OUTPUT_CSV,
            mode="a",
            header=False,
            index=False
        )

        print(f"[SUCCESS] Saved nomor {nomor}")

        time.sleep(DELAY_SECONDS)

    except Exception as e:

        print(f"[ERROR] Nomor {nomor}")
        print(e)

        time.sleep(3)

print("\nDONE")

Processing nomor 42000 sampai 42100
[SKIP] 42000 already processed
[SKIP] 42001 already processed

[PROCESSING] Nomor 42002
[SUCCESS] Saved nomor 42002

[PROCESSING] Nomor 42003
[SUCCESS] Saved nomor 42003

[PROCESSING] Nomor 42004
[SUCCESS] Saved nomor 42004

[PROCESSING] Nomor 42005
[SUCCESS] Saved nomor 42005

[PROCESSING] Nomor 42006
[ERROR] Nomor 42006
Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}

[PROCESSING] Nomor 42007
[ERROR] Nomor 42007
Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}

[PROCESSING] Nomor 42008
[ERROR] Nomor 42008
Error code: 402 - {'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inferenc